In [6]:
import json
import asyncio
import aiohttp
import os
import nest_asyncio
from typing import List, Dict, Any

nest_asyncio.apply()

# ── Configuration ─────────────────────────────────────────────
INPUT_FILE       = "filtered_training_data.json"
OUTPUT_FILE      = "dataset_fixed.json"
CHECKPOINT_FILE  = "checkpoint_filtered.json"
BATCH_SIZE       = 50
MAX_CONCURRENT   = 3
OLLAMA_URL       = "http://localhost:11434/api/generate"
MODEL            = "llama3.2:3b"
TEMPERATURE      = 0.0
MAX_TOKENS       = 512

SYSTEM_PROMPT = """You are a data cleaner. Ensure 'question' and 'output' fields are English only, keeping original meaning intact. The 'output' must be based strictly on the provided context and must be at least two complete sentences. Do not add external knowledge. Return ONLY valid JSON with keys: "question", "output". No explanation, no markdown."""

# ── File Helpers ───────────────────────────────────────────────
def load_dataset(file_path: str) -> List[Dict[str, Any]]:
    """Load JSON with automatic format detection and repair."""
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read().strip()

    # Try 1: normal JSON load
    try:
        data = json.loads(content)
        print(f"Loaded {len(data)} entries (JSON array)")
        return data
    except json.JSONDecodeError:
        pass

    # Try 2: JSONL (one object per line)
    try:
        data, errors = [], []
        for i, line in enumerate(content.splitlines(), 1):
            line = line.strip()
            if not line:
                continue
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                errors.append((i, str(e), line[:80]))
        if data:
            if errors:
                print(f"JSONL: loaded {len(data)} entries, skipped {len(errors)} bad lines:")
                for lineno, err, preview in errors[:5]:
                    print(f"  Line {lineno}: {err} | {preview}")
            else:
                print(f"Loaded {len(data)} entries (JSONL)")
            return data
    except Exception:
        pass

    # Try 3: truncated JSON array — find last complete entry
    try:
        last_brace = content.rfind('},')
        if last_brace != -1:
            repaired = content[:last_brace + 1] + ']'
            if not repaired.startswith('['):
                repaired = '[' + repaired
            data = json.loads(repaired)
            print(f"Repaired truncated JSON: recovered {len(data)} entries")
            return data
    except json.JSONDecodeError:
        pass

    # Try 4: scan line by line and report exactly where it breaks
    print("All load strategies failed. Scanning for bad lines...")
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f, 1):
            try:
                json.loads(line.strip())
            except json.JSONDecodeError as e:
                print(f"  Bad line {i}: {e}")
                print(f"  Preview: {repr(line[:200])}")
                if i > 10:
                    print("  (showing first 10 bad lines only)")
                    break
    raise ValueError("Could not load dataset. See bad lines above.")


def save_dataset(data: List[Dict[str, Any]], file_path: str):
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def load_checkpoint() -> int:
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f).get('last_index', 0)
    return 0


def save_checkpoint(index: int):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({'last_index': index}, f)


# ── Core Fix Logic ─────────────────────────────────────────────
async def fix_entry(session: aiohttp.ClientSession, entry: Dict[str, Any]) -> Dict[str, Any]:
    context_part  = entry.get('input', '')
    question_text = context_part.split("Question: ")[-1].strip() if "Question: " in context_part else entry.get('question', '')
    current_output = entry.get('output', '')

    user_prompt = f"""Entry to fix:
- Context: {context_part}
- Current question: {question_text}
- Current output: {current_output}

Fix question and output to English only. Return JSON with "question" and "output" only."""

    payload = {
        "model": MODEL,
        "prompt": user_prompt,
        "system": SYSTEM_PROMPT,
        "stream": False,
        "temperature": TEMPERATURE,
        "options": {"num_predict": MAX_TOKENS}
    }

    response_text = ""
    try:
        async with session.post(
            OLLAMA_URL, json=payload,
            timeout=aiohttp.ClientTimeout(total=120)
        ) as resp:
            if resp.status != 200:
                entry['_fix_error'] = f"HTTP {resp.status}"
                return entry

            result = await resp.json()
            response_text = result.get('response', '').strip()

            if not response_text:
                entry['_fix_error'] = "empty response"
                return entry

            # Clean markdown fences
            response_text = response_text.replace("```json", "").replace("```", "").strip()

            # Extract JSON object even if model adds extra text
            start = response_text.find('{')
            end   = response_text.rfind('}')
            if start == -1 or end == -1:
                entry['_fix_error'] = f"no JSON found: {response_text[:100]}"
                return entry

            response_text = response_text[start:end + 1]
            fixed = json.loads(response_text)

            entry['question'] = fixed.get('question', question_text)
            entry['output']   = fixed.get('output', current_output)
            entry['_fixed']   = True

    except json.JSONDecodeError as e:
        entry['_fix_error'] = f"JSON parse error: {e} | raw: {response_text[:200]}"
    except asyncio.TimeoutError:
        entry['_fix_error'] = "timeout"
    except Exception as e:
        entry['_fix_error'] = str(e)

    return entry


# ── Batch Processor ────────────────────────────────────────────
async def process_dataset(data: List[Dict[str, Any]], start_idx: int):
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)

    async def process_one(entry):
        async with semaphore:
            if entry.get('_fixed'):
                return
            await fix_entry(session, entry)

    total_entries  = len(data)
    total_batches  = (total_entries - start_idx + BATCH_SIZE - 1) // BATCH_SIZE
    batch_num      = 0

    async with aiohttp.ClientSession() as session:
        for i in range(start_idx, total_entries, BATCH_SIZE):
            batch     = data[i:i + BATCH_SIZE]
            batch_num += 1

            await asyncio.gather(*[process_one(e) for e in batch])

            done        = i + len(batch)
            errors      = sum(1 for e in batch if '_fix_error' in e)
            fixed       = sum(1 for e in batch if e.get('_fixed'))
            skipped     = len(batch) - errors - fixed

            save_checkpoint(done)
            save_dataset(data, OUTPUT_FILE)

            print(
                f"Batch {batch_num}/{total_batches} | "
                f"Entries {done}/{total_entries} | "
                f"Fixed: {fixed} | Skipped: {skipped} | Errors: {errors}"
            )

    # ── Final summary ──
    total_fixed   = sum(1 for e in data if e.get('_fixed'))
    total_errors  = sum(1 for e in data if '_fix_error' in e)
    total_skipped = total_entries - total_fixed - total_errors

    print("\n── Done ─────────────────────────────────")
    print(f"  Total entries : {total_entries}")
    print(f"  Fixed         : {total_fixed}")
    print(f"  Already clean : {total_skipped}")
    print(f"  Errors        : {total_errors}")

    if total_errors:
        print("\n  Sample errors:")
        errored = [e for e in data if '_fix_error' in e]
        for e in errored[:5]:
            print(f"    {e.get('_fix_error')} | q: {str(e.get('question',''))[:60]}")


# ── Entry Point ────────────────────────────────────────────────
async def main():
    if not os.path.exists(INPUT_FILE):
        print(f"Input file '{INPUT_FILE}' not found.")
        return

    data       = load_dataset(INPUT_FILE)
    start_idx  = load_checkpoint()
    print(f"Resuming from index {start_idx} / {len(data)}\n")

    if start_idx >= len(data):
        print("All entries already processed.")
        return

    await process_dataset(data, start_idx)

await main()

Repaired truncated JSON: recovered 1076 entries
Resuming from index 0 / 1076

Batch 1/22 | Entries 50/1076 | Fixed: 50 | Skipped: 0 | Errors: 0
Batch 2/22 | Entries 100/1076 | Fixed: 50 | Skipped: 0 | Errors: 0
Batch 3/22 | Entries 150/1076 | Fixed: 49 | Skipped: 0 | Errors: 1
Batch 4/22 | Entries 200/1076 | Fixed: 50 | Skipped: 0 | Errors: 0
Batch 5/22 | Entries 250/1076 | Fixed: 50 | Skipped: 0 | Errors: 0
Batch 6/22 | Entries 300/1076 | Fixed: 49 | Skipped: 0 | Errors: 1
Batch 7/22 | Entries 350/1076 | Fixed: 50 | Skipped: 0 | Errors: 0
Batch 8/22 | Entries 400/1076 | Fixed: 50 | Skipped: 0 | Errors: 0
Batch 9/22 | Entries 450/1076 | Fixed: 48 | Skipped: 0 | Errors: 2
Batch 10/22 | Entries 500/1076 | Fixed: 47 | Skipped: 0 | Errors: 3
Batch 11/22 | Entries 550/1076 | Fixed: 49 | Skipped: 0 | Errors: 1
Batch 12/22 | Entries 600/1076 | Fixed: 49 | Skipped: 0 | Errors: 1
Batch 13/22 | Entries 650/1076 | Fixed: 47 | Skipped: 0 | Errors: 3
Batch 14/22 | Entries 700/1076 | Fixed: 49 | Ski

In [7]:
import json
import asyncio
import aiohttp
import os
import re
import nest_asyncio
from typing import List, Dict, Any

nest_asyncio.apply()

# ── Configuration ─────────────────────────────────────────────
INPUT_FILE      = "filtered_training_data.json"
OUTPUT_FILE     = "dataset_fixed_new.json"
CHECKPOINT_FILE = "checkpoint_filtered_new.json"
BATCH_SIZE      = 50
MAX_CONCURRENT  = 3
OLLAMA_URL      = "http://localhost:11434/api/generate"
MODEL           = "llama3.2:3b"
TEMPERATURE     = 0.0
MAX_TOKENS      = 512

DEVANAGARI = re.compile(r'[\u0900-\u097F]')

SYSTEM_PROMPT_FIX = """You are a data cleaner. Your tasks:
1. Rewrite 'question' to be natural, grammatically correct English only — no Nepali/Devanagari script.
2. Rewrite 'output' to be English only, strictly based on the provided context, minimum 2 complete sentences.
3. If the question is too garbled or nonsensical to fix, set "drop": true in your response.

Return ONLY valid JSON with keys: "question", "output", "drop" (bool). No explanation, no markdown."""

SYSTEM_PROMPT_REGEN = """You are a QnA answer generator. Given a context (in Nepali) and a question (in English):
1. Answer the question strictly using only information from the context.
2. Write the answer in English, minimum 2 complete sentences.
3. If the answer is not in the context, write: "I cannot find this information in the provided context."

Return ONLY valid JSON with keys: "output". No explanation, no markdown."""

# ── File Helpers ───────────────────────────────────────────────
def load_dataset(file_path: str) -> List[Dict[str, Any]]:
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read().strip()

    try:
        data = json.loads(content)
        print(f"Loaded {len(data)} entries (JSON array)")
        return data
    except json.JSONDecodeError:
        pass

    try:
        data, errors = [], []
        for i, line in enumerate(content.splitlines(), 1):
            line = line.strip()
            if not line:
                continue
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                errors.append((i, str(e), line[:80]))
        if data:
            if errors:
                print(f"JSONL: loaded {len(data)} entries, skipped {len(errors)} bad lines")
            else:
                print(f"Loaded {len(data)} entries (JSONL)")
            return data
    except Exception:
        pass

    try:
        last_brace = content.rfind('},')
        if last_brace != -1:
            repaired = content[:last_brace + 1] + ']'
            if not repaired.startswith('['):
                repaired = '[' + repaired
            data = json.loads(repaired)
            print(f"Repaired truncated JSON: recovered {len(data)} entries")
            return data
    except json.JSONDecodeError:
        pass

    print("All load strategies failed. Scanning for bad lines...")
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f, 1):
            try:
                json.loads(line.strip())
            except json.JSONDecodeError as e:
                print(f"  Bad line {i}: {e} | {repr(line[:200])}")
                if i > 10:
                    break
    raise ValueError("Could not load dataset.")


def save_dataset(data: List[Dict[str, Any]], file_path: str):
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def load_checkpoint() -> int:
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f).get('last_index', 0)
    return 0


def save_checkpoint(index: int):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({'last_index': index}, f)


def needs_fixing(entry: Dict[str, Any]) -> bool:
    """Check if question or output contains Devanagari."""
    return (
        DEVANAGARI.search(entry.get('question', '')) is not None or
        DEVANAGARI.search(entry.get('output', '')) is not None
    )


def extract_json(text: str) -> str:
    text = text.replace("```json", "").replace("```", "").strip()
    start = text.find('{')
    end   = text.rfind('}')
    if start != -1 and end != -1:
        return text[start:end + 1]
    return ""


# ── Ollama Call ────────────────────────────────────────────────
async def call_ollama(
    session: aiohttp.ClientSession,
    system: str,
    prompt: str
) -> str:
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "system": system,
        "stream": False,
        "temperature": TEMPERATURE,
        "options": {"num_predict": MAX_TOKENS}
    }
    async with session.post(
        OLLAMA_URL, json=payload,
        timeout=aiohttp.ClientTimeout(total=120)
    ) as resp:
        if resp.status != 200:
            raise RuntimeError(f"HTTP {resp.status}")
        result = await resp.json()
        return result.get('response', '').strip()


# ── Pass 1: Fix language ───────────────────────────────────────
async def pass1_fix_language(
    session: aiohttp.ClientSession,
    entry: Dict[str, Any]
) -> Dict[str, Any]:
    context_part   = entry.get('input', '')
    question_text  = context_part.split("Question: ")[-1].strip() if "Question: " in context_part else entry.get('question', '')
    current_output = entry.get('output', '')

    prompt = f"""Entry to fix:
- Context: {context_part}
- Current question: {question_text}
- Current output: {current_output}

Rewrite question and output to clean English. If question is too garbled to fix meaningfully, set "drop": true."""

    raw = await call_ollama(session, SYSTEM_PROMPT_FIX, prompt)
    raw = extract_json(raw)
    if not raw:
        entry['_fix_error'] = "pass1: no JSON found"
        return entry

    fixed = json.loads(raw)

    if fixed.get('drop'):
        entry['_dropped'] = True
        entry['_fixed']   = True
        return entry

    entry['question']      = fixed.get('question', question_text)
    entry['output']        = fixed.get('output', current_output)
    entry['_needs_regen']  = _answer_needs_regen(entry['question'], entry['output'])
    return entry


def _answer_needs_regen(question: str, output: str) -> bool:
    """Flag if output is vague, empty, or doesn't seem to answer the question."""
    vague_phrases = [
        "no specific requirements are mentioned",
        "not mentioned",
        "no mentioned requirements",
        "cannot determine",
        "i cannot find",
    ]
    output_lower = output.lower()
    if len(output.strip()) < 30:
        return True
    if any(p in output_lower for p in vague_phrases):
        return True
    return False


# ── Pass 2: Regenerate answer from context ─────────────────────
async def pass2_regen_answer(
    session: aiohttp.ClientSession,
    entry: Dict[str, Any]
) -> Dict[str, Any]:
    context_part = entry.get('input', '')
    question     = entry.get('question', '')

    prompt = f"""Context: {context_part}
Question: {question}

Answer the question strictly from the context in English, minimum 2 sentences."""

    raw = await call_ollama(session, SYSTEM_PROMPT_REGEN, prompt)
    raw = extract_json(raw)
    if not raw:
        entry['_fix_error'] = "pass2: no JSON found"
        return entry

    result = json.loads(raw)
    entry['output'] = result.get('output', entry.get('output', ''))
    entry['_fixed'] = True
    entry.pop('_needs_regen', None)
    return entry


# ── Combined entry processor ───────────────────────────────────
async def fix_entry(
    session: aiohttp.ClientSession,
    entry: Dict[str, Any]
) -> Dict[str, Any]:
    try:
        # Pass 1: fix language
        entry = await pass1_fix_language(session, entry)

        if entry.get('_fix_error') or entry.get('_dropped'):
            return entry

        # Pass 2: regenerate answer if needed
        if entry.get('_needs_regen'):
            entry = await pass2_regen_answer(session, entry)
        else:
            entry['_fixed'] = True

    except json.JSONDecodeError as e:
        entry['_fix_error'] = f"JSON parse: {e}"
    except asyncio.TimeoutError:
        entry['_fix_error'] = "timeout"
    except Exception as e:
        entry['_fix_error'] = str(e)

    return entry


# ── Batch Processor ────────────────────────────────────────────
async def process_dataset(data: List[Dict[str, Any]], start_idx: int):
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)

    async def process_one(entry):
        async with semaphore:
            if entry.get('_fixed'):
                return
            if not needs_fixing(entry):
                entry['_fixed'] = True  # already clean, mark and skip
                return
            await fix_entry(session, entry)

    total_batches = (len(data) - start_idx + BATCH_SIZE - 1) // BATCH_SIZE
    batch_num     = 0

    async with aiohttp.ClientSession() as session:
        for i in range(start_idx, len(data), BATCH_SIZE):
            batch     = data[i:i + BATCH_SIZE]
            batch_num += 1

            await asyncio.gather(*[process_one(e) for e in batch])

            done    = i + len(batch)
            errors  = sum(1 for e in batch if '_fix_error' in e)
            dropped = sum(1 for e in batch if e.get('_dropped'))
            regen   = sum(1 for e in batch if e.get('_fixed') and not e.get('_dropped'))

            save_checkpoint(done)
            save_dataset(data, OUTPUT_FILE)

            print(
                f"Batch {batch_num}/{total_batches} | "
                f"{done}/{len(data)} entries | "
                f"Fixed: {regen} | Dropped: {dropped} | Errors: {errors}"
            )

    # ── Summary ──
    total_fixed   = sum(1 for e in data if e.get('_fixed') and not e.get('_dropped'))
    total_dropped = sum(1 for e in data if e.get('_dropped'))
    total_errors  = sum(1 for e in data if '_fix_error' in e)

    print("\n── Done ──────────────────────────────────")
    print(f"  Total   : {len(data)}")
    print(f"  Fixed   : {total_fixed}")
    print(f"  Dropped : {total_dropped}")
    print(f"  Errors  : {total_errors}")

    if total_errors:
        print("\n  Sample errors:")
        for e in [x for x in data if '_fix_error' in x][:5]:
            print(f"    {e['_fix_error']} | q: {str(e.get('question',''))[:60]}")


# ── Entry Point ────────────────────────────────────────────────
async def main():
    if not os.path.exists(INPUT_FILE):
        print(f"Input file '{INPUT_FILE}' not found.")
        return

    data      = load_dataset(INPUT_FILE)
    start_idx = load_checkpoint()
    print(f"Resuming from {start_idx} / {len(data)}\n")

    if start_idx >= len(data):
        print("All entries already processed.")
        return

    await process_dataset(data, start_idx)

await main()

Exception ignored in: <coroutine object main at 0x7de5ec1ee540>
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import__'
Exception ignored in: <coroutine object main at 0x7de5ec1ee540>
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import__'
Exception ignored in: <coroutine object main at 0x7de5ecf94a40>
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import__'


Repaired truncated JSON: recovered 1076 entries
Resuming from 0 / 1076

Batch 1/22 | 50/1076 entries | Fixed: 50 | Dropped: 0 | Errors: 0
Batch 2/22 | 100/1076 entries | Fixed: 50 | Dropped: 0 | Errors: 0
Batch 3/22 | 150/1076 entries | Fixed: 50 | Dropped: 0 | Errors: 0
Batch 4/22 | 200/1076 entries | Fixed: 50 | Dropped: 0 | Errors: 0
Batch 5/22 | 250/1076 entries | Fixed: 50 | Dropped: 0 | Errors: 0
Batch 6/22 | 300/1076 entries | Fixed: 50 | Dropped: 0 | Errors: 0
Batch 7/22 | 350/1076 entries | Fixed: 50 | Dropped: 0 | Errors: 0
Batch 8/22 | 400/1076 entries | Fixed: 50 | Dropped: 0 | Errors: 0
Batch 9/22 | 450/1076 entries | Fixed: 50 | Dropped: 0 | Errors: 0
Batch 10/22 | 500/1076 entries | Fixed: 50 | Dropped: 0 | Errors: 0
Batch 11/22 | 550/1076 entries | Fixed: 50 | Dropped: 0 | Errors: 0
Batch 12/22 | 600/1076 entries | Fixed: 50 | Dropped: 0 | Errors: 0
Batch 13/22 | 650/1076 entries | Fixed: 49 | Dropped: 0 | Errors: 1
Batch 14/22 | 700/1076 entries | Fixed: 50 | Dropped: 